# 05 - Training and comparing models

**Split method used for everything here:** walk-forward validation on `development.csv` (5 time-ordered blocks, 4 rounds; see notebook 03). `test.csv` is never loaded in this notebook. Every model uses **class weighting**, carried forward from notebook 04.

## Importing our own shared code

`prepare_round` and `aucpr_low_recall` (written in notebook 04) now live in a real Python file, `src/walk_forward.py`, a **module**, so every notebook from here on runs exactly the same code instead of its own copy.

Python looks for modules in a list of folders, `sys.path`. This notebook lives in `notebooks/`, so `sys.path.append("..")` adds the project folder (one level up) to that list; `from src.walk_forward import ...` then finds `src/walk_forward.py` and brings in the named pieces. (The alternative, copying the functions into each notebook, would leave the same code in several places, where a fix to one copy could silently miss another.)

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from src.walk_forward import aucpr_low_recall, feature_columns, prepare_round

print(sys.executable)

development = pd.read_csv("../data/processed/development.csv")
print(development.shape)

recall_limit = 0.2

C:\Users\RevathyHarichandran\Desktop\portfolio projects\Fraud_detection\.venv\Scripts\python.exe


(226980, 34)


## Part 1: Why Amount needs scaling, and what "fit only on training" means

### Step 1: The size gap

The **standard deviation** (typical spread) of each feature in round 1's learning blocks. Logistic Regression gives every feature a weight, and by default gently holds every weight back from getting too large, with the same pressure on all of them. A feature measured in much bigger units only needs a tiny weight, so that shared pressure ends up treating features differently because of their units, not their usefulness. Very different sizes also make the model's step-by-step learning lurch in one direction and crawl in the others.

In [2]:
round_1_learn = development[development["block"] <= 1]
spreads = round_1_learn[feature_columns].std()

v_spreads = spreads.drop("Amount")
print(f"V columns: smallest spread {v_spreads.min():.3f}, largest {v_spreads.max():.3f}")
print(f"Amount:    spread {spreads['Amount']:.3f}")
print(f"Amount's spread is {spreads['Amount'] / v_spreads.max():.0f} times the largest V column's")

V columns: smallest spread 0.339, largest 1.880
Amount:    spread 240.780
Amount's spread is 128 times the largest V column's


`drop("Amount")` leaves out the Amount entry, so the V columns can be summarised on their own.

### Step 2: What the scaler learns in each round, and what it never sees

A **StandardScaler** learns two numbers, the mean and the standard deviation (it divides by the number of rows, `ddof=0`, checked with a small test), and turns each Amount into "how many standard deviations from that mean". **Fitting** = calculating those two numbers; **applying** = using them to rescale.

In every round they are calculated **from the learning blocks only** and then applied unchanged to the check block. The table shows, per round, the two numbers the scaler learned next to the check block's *own* mean and spread: the information a scaler fitted on the check block too would have quietly taken from "the future".

Then a check: after scaling, the learning blocks' Amount must have mean 0 and spread 1 exactly (by construction), while the check block's will only be *near* those, the honest sign that the scaler never saw it.

In [3]:
for round_number in range(1, 5):
    learn = development[development["block"] <= round_number]
    check = development[(development["block"] == round_number + 1) & (~development["in_gap"])]
    print(f"round {round_number}: scaler learned mean {learn['Amount'].mean():7.2f}, spread {learn['Amount'].std(ddof=0):7.2f}"
          f"  | check block's own mean {check['Amount'].mean():7.2f}, spread {check['Amount'].std(ddof=0):7.2f}")

    learn_X, learn_y, check_X, check_y = prepare_round(development, round_number, feature_columns)
    print(f"         after scaling: learning mean {learn_X['Amount'].mean():.4f}, spread {learn_X['Amount'].std(ddof=0):.4f}"
          f"  | check mean {check_X['Amount'].mean():.4f}, spread {check_X['Amount'].std(ddof=0):.4f}")

round 1: scaler learned mean   91.05, spread  240.78  | check block's own mean  107.04, spread  288.72
         after scaling: learning mean -0.0000, spread 1.0000  | check mean 0.0664, spread 1.1991


round 2: scaler learned mean   99.01, spread  268.46  | check block's own mean   76.35, spread  199.78


         after scaling: learning mean -0.0000, spread 1.0000  | check mean -0.0844, spread 0.7442
round 3: scaler learned mean   91.72, spread  248.76  | check block's own mean   79.23, spread  245.80


         after scaling: learning mean -0.0000, spread 1.0000  | check mean -0.0502, spread 0.9881


round 4: scaler learned mean   88.60, spread  247.82  | check block's own mean  100.00, spread  262.36


         after scaling: learning mean -0.0000, spread 1.0000  | check mean 0.0460, spread 1.0587


### Step 3: Experiment, does scaling Amount actually matter here?

In every round, the same class-weighted Logistic Regression (`max_iter=1000`, `class_weight="balanced"`, as in notebook 04) is trained twice on exactly the same rows: once with Amount scaled (from `prepare_round`), once with Amount left in its raw units. Only Amount differs; the V columns are identical in both.

Recorded for each:
- **low-recall AUCPR** and **average precision** on the check block (both explained in notebook 04);
- **learning steps used** (`n_iter_`, how many steps the model needed to settle; 1,000 means it hit the limit without settling);
- **warnings** raised while learning. `warnings.catch_warnings(record=True)` collects any warnings into a list instead of printing them, so they can be counted. `with` opens that collecting "block" for the indented lines only, and closes it again afterwards. `simplefilter("always")` makes sure every warning is recorded, even repeats;
- **Amount's weight** (`coef_`, the model's weight for each feature, in feature order; Amount is the last, `[0, -1]`).

Both versions' check-block scores are pooled across the four rounds for a paired comparison afterwards.

In [4]:
import warnings

experiment_rows = []
pooled_answers = []
pooled_scaled_scores = []
pooled_raw_scores = []

for round_number in range(1, 5):
    learn_X, learn_y, check_X, check_y = prepare_round(development, round_number, feature_columns)

    # The same rows with Amount put back in its raw, unscaled units.
    learn = development[development["block"] <= round_number]
    check = development[(development["block"] == round_number + 1) & (~development["in_gap"])]
    learn_raw_X = learn[feature_columns]
    check_raw_X = check[feature_columns]

    pooled_answers.append(check_y.to_numpy())

    versions = [["scaled Amount", learn_X, check_X], ["raw Amount", learn_raw_X, check_raw_X]]
    for version in versions:
        version_name = version[0]
        version_learn_X = version[1]
        version_check_X = version[2]

        model = LogisticRegression(max_iter=1000, class_weight="balanced")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(version_learn_X, learn_y)
        scores = model.predict_proba(version_check_X)[:, 1]

        if version_name == "scaled Amount":
            pooled_scaled_scores.append(scores)
        else:
            pooled_raw_scores.append(scores)

        experiment_rows.append({
            "round": round_number,
            "version": version_name,
            "low_recall_aucpr": aucpr_low_recall(check_y, scores, recall_limit),
            "average_precision": average_precision_score(check_y, scores),
            "learning_steps": model.n_iter_[0],
            "warnings": len(caught),
            "amount_weight": model.coef_[0, -1],
        })

experiment_table = pd.DataFrame(experiment_rows)
print(experiment_table.round(4).to_string(index=False))

 round       version  low_recall_aucpr  average_precision  learning_steps  warnings  amount_weight
     1 scaled Amount            0.8292             0.5896              69         0        -1.1648
     1    raw Amount            0.8292             0.5826            1000         1        -0.0051
     2 scaled Amount            0.8854             0.5823              59         0        -0.2735
     2    raw Amount            0.8854             0.5814            1000         1        -0.0010
     3 scaled Amount            1.0000             0.8413              45         0         1.3013
     3    raw Amount            1.0000             0.8367            1000         1         0.0047
     4 scaled Amount            1.0000             0.7821              40         0         1.7222
     4    raw Amount            1.0000             0.7810             832         0         0.0071


In the cell above, `versions` is a list of small lists, each holding a name and its learning and check features, so one loop can run both versions identically. `model.n_iter_[0]` is the step count (scikit-learn stores it inside a list, so `[0]` takes it out).

### Step 4: Is any difference real? Paired bootstrap

Both versions were scored on exactly the same pooled check rows (250 fraud), so the paired-difference bootstrap applies (same as notebook 04: 1,000 resamples, seed 42). Scaled minus raw: an interval entirely above zero would mean scaling genuinely helped; entirely below, that it genuinely hurt; including zero, no proven difference.

In [5]:
all_answers = np.concatenate(pooled_answers)
all_scaled = np.concatenate(pooled_scaled_scores)
all_raw = np.concatenate(pooled_raw_scores)

number_of_resamples = 1000
rng = np.random.default_rng(42)
number_of_rows = len(all_answers)

low_recall_differences = []
ap_differences = []
for resample_number in range(number_of_resamples):
    positions = rng.integers(0, number_of_rows, size=number_of_rows)
    answers = all_answers[positions]
    low_recall_differences.append(aucpr_low_recall(answers, all_scaled[positions], recall_limit)
                                  - aucpr_low_recall(answers, all_raw[positions], recall_limit))
    ap_differences.append(average_precision_score(answers, all_scaled[positions])
                          - average_precision_score(answers, all_raw[positions]))

for name, scores in [["scaled Amount", all_scaled], ["raw Amount", all_raw]]:
    print(f"{name:14s} pooled low-recall AUCPR {aucpr_low_recall(all_answers, scores, recall_limit):.4f}"
          f"  average precision {average_precision_score(all_answers, scores):.4f}")
print()
print(f"scaled minus raw, low-recall AUCPR: 95% CI {np.percentile(low_recall_differences, 2.5):+.4f} to {np.percentile(low_recall_differences, 97.5):+.4f}")
print(f"scaled minus raw, average precision: 95% CI {np.percentile(ap_differences, 2.5):+.4f} to {np.percentile(ap_differences, 97.5):+.4f}")

scaled Amount  pooled low-recall AUCPR 0.9966  average precision 0.6923
raw Amount     pooled low-recall AUCPR 0.9984  average precision 0.6951

scaled minus raw, low-recall AUCPR: 95% CI -0.0064 to +0.0000
scaled minus raw, average precision: 95% CI -0.0071 to +0.0011


In the cell above, `for name, scores in [[...], [...]]:` takes each small list in turn and unpacks its two items straight into `name` and `scores`.

**What this shows:**
- **The size gap:** Amount's spread (240.78) is 128 times the largest V column's (1.880).
- **Fit only on training, made concrete:** the scaler's learned mean and spread differ from each check block's own (round 1: learned mean 91.05, check block's 107.04; round 2: 99.01 against 76.35). A scaler fitted on the check block too would have absorbed that future information. After scaling, the learning blocks sit at exactly mean 0 and spread 1, while the check blocks only land near them (for example spread 0.7442 in round 2): the visible sign the scaler never saw them.
- **Scaling made no proven difference to performance here:** pooled, scaled minus raw is −0.0064 to +0.0000 on low-recall AUCPR and −0.0071 to +0.0011 on average precision; both intervals include zero.
- **Scaling made a large difference to learning:** with scaled Amount the model settled in 40 to 69 steps with no warnings; with raw Amount it hit the 1,000-step limit without settling in rounds 1 to 3 (one "did not finish" warning each) and needed 832 steps in round 4.
- **The model learned the same thing about Amount either way, in different units:** round 1's raw weight −0.0051 per unit of money × spread 240.78 ≈ −1.23, close to the scaled weight −1.1648 per standard deviation.
- **Why the weight-holding-back problem barely showed:** with 45,000 to 180,000 learning rows, the default holding-back (`C=1.0`) is weak next to the evidence in the data.
- **Decision: keep scaling Amount**, because an unfinished model is unreliable even when it happens to land close, and the rules require it for scale-sensitive models; not because it improved the score here, which it did not measurably.

## Part 2: Logistic Regression

Uses **class weighting**, carried forward from notebook 04. That was a practical tie-break with SMOTE, not a proven best.

### What Logistic Regression is doing

A **learned points system**:
1. Every feature gets a **weight** (points per unit), 29 in all, plus one starting number, the **intercept**.
2. For each transaction: multiply each feature by its weight, add them up, add the intercept.
3. Squash that total into a score between 0 and 1 with the S-shaped **sigmoid** curve: big negative totals land near 0 (looks genuine), big positive ones near 1 (looks like fraud), a total of 0 lands at 0.5.
4. **Learning:** start all weights at zero and adjust them step by step so the scores move closer to the real answers. With class weighting, each mistake on a fraud case counts about 568 times as much as one on a genuine case.

**Regularisation (`C`):** an extra penalty discourages very large weights, so the model doesn't over-rely on one feature or memorise quirks of the learning data. `C=1.0` is the default; a smaller C holds back harder.

**Its limit:** it only adds up weighted features, so it draws a straight dividing line (in 29 dimensions). It can't learn combinations like "fraud if V14 is low *and* Amount is high, but not otherwise"; that is where tree models come in.

**Scores aren't probabilities after class weighting:** fraud mistakes counted 568 times more during learning, so scores come out inflated (0.9 does not mean "90% chance of fraud"). They still *rank* transactions, which is all the metrics here use; it matters again when turning scores into decisions in Phase 7.

### Settings

`class_weight="balanced"`, `max_iter=1000` (room to finish; scaled data needed 40 to 69 steps), `C=1.0` (default), solver `lbfgs` (default step-by-step method), standard L2 holding-back (default). **Not searching for the best settings is deliberate:** the goal is an honest, simple baseline, and tuning `C` would mean more comparisons on the same 250 check fraud cases, risking a value that happens to suit those particular cases.

### Running it

The shared module now also holds `make_model` (builds an untrained model by name, so every model is built one way) and `run_walk_forward` (runs all 4 rounds for one model: prepare the round with the scaler fitted on learning blocks only, train, score the check block; returns the round-by-round table, the pooled answers and scores, and the 4 trained models), plus `bootstrap_metrics` (the 1,000-resample bootstrap used so far, now written once so every model is resampled on the same rows).

In [6]:
import time

from src.walk_forward import bootstrap_metrics, run_walk_forward

start = time.time()
lr_rounds, pooled_answers, lr_scores, lr_models = run_walk_forward(development, "Logistic Regression", recall_limit, feature_columns)
lr_minutes = (time.time() - start) / 60

print("Logistic Regression, round by round:")
print(lr_rounds.round(4).to_string(index=False))
print()

lr_bootstrap = bootstrap_metrics(pooled_answers, {"Logistic Regression": lr_scores}, recall_limit, 1000, 42)
lr_low_recall = lr_bootstrap["Logistic Regression"]["low_recall"]
lr_average_precision = lr_bootstrap["Logistic Regression"]["average_precision"]

print(f"pooled check rows: {len(pooled_answers):,}, fraud: {pooled_answers.sum()}")
print(f"pooled low-recall AUCPR {aucpr_low_recall(pooled_answers, lr_scores, recall_limit):.4f}"
      f"  95% CI {np.percentile(lr_low_recall, 2.5):.4f} to {np.percentile(lr_low_recall, 97.5):.4f}")
print(f"pooled average precision {average_precision_score(pooled_answers, lr_scores):.4f}"
      f"  95% CI {np.percentile(lr_average_precision, 2.5):.4f} to {np.percentile(lr_average_precision, 97.5):.4f}")

Logistic Regression, round by round:
              model  round  check_fraud  low_recall_aucpr  average_precision  learning_steps
Logistic Regression      1           66            0.8292             0.5896              69
Logistic Regression      2           46            0.8854             0.5823              59
Logistic Regression      3           88            1.0000             0.8413              45
Logistic Regression      4           50            1.0000             0.7821              40



pooled check rows: 176,175, fraud: 250
pooled low-recall AUCPR 0.9966  95% CI 0.9831 to 1.0000
pooled average precision 0.6923  95% CI 0.6310 to 0.7492


### A look inside: the biggest weights

The round 4 model (the one with the most learning data). `coef_[0]` holds its 29 weights in feature order; `pd.Series(..., index=feature_columns)` labels each with its feature name. Sorting by the weight's size with the sign dropped (`abs()`, then `sort_values(ascending=False)`, largest first) shows which features the "points system" leans on most. A positive weight pushes towards fraud, a negative one towards genuine.

**Honest limit:** these are anonymized V columns, so the explanation stops at "this column matters a lot". This is a peek for understanding, not a feature-selection step.

In [7]:
round_4_model = lr_models[3]
weights = pd.Series(round_4_model.coef_[0], index=feature_columns)
weight_sizes = weights.abs().sort_values(ascending=False)

print(f"intercept: {round_4_model.intercept_[0]:.4f}")
print("five largest weights (by size):")
for feature_name in weight_sizes.index[:5]:
    print(f"  {feature_name:7s} {weights[feature_name]:+.4f}")

intercept: -3.5781
five largest weights (by size):
  Amount  +1.7222
  V14     -1.0555
  V12     -0.9356
  V20     -0.8108
  V4      +0.8010


**What this shows:**
- **Logistic Regression with class weighting, pooled over 250 check fraud cases:** low-recall AUCPR 0.9966 (95% CI 0.9831 to 1.0000), average precision 0.6923 (0.6310 to 0.7492). These match notebook 04's class-weighting results exactly, confirming the shared module behaves identically.
- **Round by round:** low-recall AUCPR 0.8292, 0.8854, 1.0000, 1.0000; average precision 0.5896, 0.5823, 0.8413, 0.7821. Both improve as rounds learn from more data, and the headline metric hits its ceiling in rounds 3 and 4, as in notebook 04.
- **Inside the round 4 model:** the intercept is −3.5781, so a transaction with every feature at its average scores sigmoid(−3.58), about 0.03. The largest weights are Amount (+1.72: larger amounts push towards fraud), V14 (−1.06), V12 (−0.94), V20 (−0.81) and V4 (+0.80).
- **Two honest cautions about those weights:** (1) the V columns are on different scales (spreads from about 0.3 to 1.9), so a weight per unit isn't a perfectly fair measure of importance across them; (2) Amount's weight changes a lot between rounds (−1.16, −0.27, +1.30, +1.72 in Part 1's experiment, the same models), so "larger amounts mean fraud" is not a stable lesson, which fits notebook 02's finding that fraud amounts are polarised (both more tiny and more large ones).

## Part 3: Random Forest

### What it does differently

**A decision tree** is a flowchart of yes/no questions learned from the data ("Is V14 below −4.2? If yes, is Amount above 0.8?..."). Each question is chosen to split transactions into groups that are as pure as possible (mostly fraud on one side, mostly genuine on the other), until each final group, a **leaf**, is decided.

- **Unlike Logistic Regression**, which adds up points along one straight dividing line, a tree asks questions one after another, so it can learn **combinations** ("fraud if V14 is very low *and* the amount is large, but genuine if the amount is tiny"), the kind of polarised pattern Amount showed in notebook 02. Trees also don't need scaling: "is Amount above 185?" splits the same transactions whatever the units.
- **One fully grown tree over-fits:** it memorises the learning data, quirks and all.
- **A forest fixes that:** 100 trees vote, made different on purpose: each learns from its own random sample of rows drawn with replacement (**bagging**), and at every question each tree may only consider a random handful of features (about 5 of 29, the square root). They make different mistakes, so averaging the votes cancels much of the noise. A transaction's score is the forest's average verdict.

### Settings (defaults confirmed in the installed scikit-learn 1.9.1)

| Setting | Value | Reason |
|---|---|---|
| `n_estimators` | 100 (default) | common, stable size; more rarely hurts but costs time |
| `max_depth` | None (default) | trees grow until leaves are decided; the averaging controls over-fitting |
| `min_samples_leaf` | 1 (default) | same reasoning |
| `max_features` | "sqrt" (default) | about 5 of 29 features per question, which keeps trees different |
| `bootstrap` | True (default) | each tree learns from its own random sample of rows |
| `class_weight` | "balanced" | carried forward; identical weights to Logistic Regression, so the comparison is about the model (the alternative, "balanced_subsample", recalculates weights per tree) |
| `random_state` | 42 | default `None` gives different trees every run |
| `n_jobs` | −1 | uses all processor cores; changes speed, never results |

**Not tuning is deliberate:** searching depth, leaf size or tree count on the same 250 check fraud cases would risk picking whatever suits those particular cases.

**Its scores tie a lot:** a forest's score is an average of 100 trees' verdicts, so scores come in coarse steps and many transactions share exactly the same score. Tied transactions move up or down the ranking together, which can make the precision-recall curve blocky. The number of distinct score values is reported below. `np.unique` lists each different value once; `len` counts them.

The forest runs through exactly the same rounds, scaler and check blocks as Logistic Regression (`run_walk_forward`, now with a `"Random Forest"` option in `make_model`). `time.time()` (notebook 04) measures how long the four rounds take.

In [8]:
start = time.time()
rf_rounds, rf_answers, rf_scores, rf_models = run_walk_forward(development, "Random Forest", recall_limit, feature_columns)
rf_minutes = (time.time() - start) / 60
print(f"Random Forest, all 4 rounds: {rf_minutes:.1f} minutes")

# Both models must have been scored on exactly the same pooled rows.
print("same pooled answers as Logistic Regression:", np.array_equal(rf_answers, pooled_answers))
print()

side_by_side = pd.concat([lr_rounds, rf_rounds])
print("Low-recall AUCPR by round:")
print(side_by_side.pivot(index="model", columns="round", values="low_recall_aucpr").round(4))
print()
print("Average precision by round:")
print(side_by_side.pivot(index="model", columns="round", values="average_precision").round(4))

Random Forest, all 4 rounds: 0.3 minutes
same pooled answers as Logistic Regression: True

Low-recall AUCPR by round:
round                     1       2       3       4
model                                              
Logistic Regression  0.8292  0.8854  1.0000  1.0000
Random Forest        1.0000  0.8918  0.9604  0.9917

Average precision by round:
round                     1       2       3       4
model                                              
Logistic Regression  0.5896  0.5823  0.8413  0.7821
Random Forest        0.8057  0.6724  0.7878  0.7482


`np.array_equal` checks two lists of numbers are identical, value by value: proof both models were scored on exactly the same transactions, which the paired comparison later depends on.

### Pooled, side by side

Both models bootstrapped together (`bootstrap_metrics`, 1,000 resamples, seed 42), so each resample scores them on the same rows. Each gets its own 95% interval here; the paired-difference comparison of all three models comes after XGBoost.

In [9]:
scores_by_model = {"Logistic Regression": lr_scores, "Random Forest": rf_scores}
two_model_bootstrap = bootstrap_metrics(pooled_answers, scores_by_model, recall_limit, 1000, 42)

for model_name in scores_by_model:
    scores = scores_by_model[model_name]
    low_recall_list = two_model_bootstrap[model_name]["low_recall"]
    ap_list = two_model_bootstrap[model_name]["average_precision"]
    print(f"{model_name}")
    print(f"  low-recall AUCPR  {aucpr_low_recall(pooled_answers, scores, recall_limit):.4f}"
          f"  95% CI {np.percentile(low_recall_list, 2.5):.4f} to {np.percentile(low_recall_list, 97.5):.4f}")
    print(f"  average precision {average_precision_score(pooled_answers, scores):.4f}"
          f"  95% CI {np.percentile(ap_list, 2.5):.4f} to {np.percentile(ap_list, 97.5):.4f}")
    print(f"  distinct score values: {len(np.unique(scores)):,} (out of {len(scores):,} transactions)")

Logistic Regression
  low-recall AUCPR  0.9966  95% CI 0.9831 to 1.0000
  average precision 0.6923  95% CI 0.6310 to 0.7492
  distinct score values: 173,055 (out of 176,175 transactions)
Random Forest
  low-recall AUCPR  0.9595  95% CI 0.8958 to 0.9976
  average precision 0.7581  95% CI 0.6977 to 0.8101
  distinct score values: 93 (out of 176,175 transactions)


**What this shows** (no winner called yet; the paired comparison of all three models comes after XGBoost):
- **Mixed, not simply better or worse.** Random Forest is clearly stronger in the early rounds (round 1 average precision 0.8057 against 0.5896; low-recall AUCPR 1.0000 against 0.8292) and weaker in rounds 3 and 4 on both metrics.
- **Pooled (250 fraud):** on the headline metric Logistic Regression is ahead (0.9966, 95% CI 0.9831 to 1.0000, against 0.9595, 0.8958 to 0.9976); on average precision Random Forest is ahead (0.7581, 0.6977 to 0.8101, against 0.6923, 0.6310 to 0.7492). The models' own intervals overlap on both, so neither lead is proven yet.
- **Ties:** Logistic Regression produced 173,055 distinct scores for 176,175 transactions; Random Forest only **93**. Large groups of transactions share each forest score, even at the very top, so real fraud can't be ranked ahead of genuine transactions within a group. That plausibly explains part of the forest's lower pooled headline score (not separately tested).
- **Pooled against mean of rounds disagree again:** averaged round by round, Random Forest is ahead on the headline metric (about 0.961 against 0.929); pooled, it is behind. Pooling ranks differently pitched round models together, which matters even more with only 93 score steps.
- **Class weighting does less for fully grown trees:** leaves are grown until decided, so the weights mostly change which questions are asked, not what the leaves conclude.
- **Speed:** all four rounds took 0.3 minutes with all processor cores.

## Part 4: XGBoost

### What makes it different from Random Forest

Both are made of many decision trees; the difference is **how the trees work together**.
- **Random Forest:** a committee voting independently. Each tree is grown separately on its own random sample and tries to solve the whole problem alone; then they vote.
- **XGBoost (gradient boosting):** a team correcting each other, one after another. The first tree makes a rough attempt; the second learns specifically to fix **where the first got it wrong**; the third fixes what the first two together still get wrong; and so on. A transaction's score is the **sum of all the trees' small corrections**. (Like one essay passed through 100 editors, each fixing what the previous ones left.)
- **Shallow trees:** each tree only fixes part of the remaining error, so it is kept a few questions deep, not grown until every leaf is decided.
- **Careful steps:** each tree's correction is scaled down by a **learning rate**, so no single tree over-corrects.
- **Built-in holding-back:** a penalty on overly large corrections, similar in spirit to Logistic Regression's `C`.
- **Smoother scores:** adding up 100 small corrections gives far fewer ties than the forest's 93 values.

The literature review's most recent source (Karlsruhe, 2025) found XGBoost performed strongly on this exact dataset, which makes it the expected favourite here; the results below test that rather than assume it.

### Settings (XGBoost 3.4.1)

Its settings list shows `None` for most values, meaning "use the built-in default". The real defaults were confirmed by training a tiny model and reading the settings it actually used, then written out in `make_model` so they are visible:

| Setting | Value | Meaning |
|---|---|---|
| `n_estimators` | 100 (default) | trees built one after another |
| `learning_rate` | 0.3 (default) | each correction scaled to 30% |
| `max_depth` | 6 (default) | each tree at most 6 questions deep |
| `min_child_weight` | 1 (default) | smallest (weighted) group a leaf may hold |
| `reg_lambda` | 1 (default) | holding-back on correction size; `reg_alpha` and `gamma` (two more options) stay at 0 |
| `subsample`, `colsample_bytree` | 1 (default) | each tree sees all rows and features (no random sampling, unlike the forest) |
| `scale_pos_weight` | genuine ÷ fraud in that round's learning blocks | class weighting, the same ratio as "balanced" in the other two models; calculated from learning data only |
| `random_state`, `n_jobs` | 42, −1 | reproducible; all processor cores (speed only) |

(Alternative to `scale_pos_weight`: per-row weights with the same effect; `scale_pos_weight` is XGBoost's standard way. `make_model` now also receives the round's learning answers to calculate it; the other two models ignore them.)

### Reasonable settings, not the best possible ones: said plainly

**All three models use reasonable, explained default settings, not a search for the absolute best ones.** That is a fair choice here because:
1. **The question is which kind of model suits this problem**, not how high one number can be pushed. Standard settings for all three compare them fairly; tuning one more than another would muddy that.
2. **Only 250 fraud cases judge every choice.** Searching many setting combinations and keeping the best is itself choosing on the data, and with so few fraud cases the "best" settings would partly be the ones that happened to suit these particular cases, making the score look better than it truly is.
3. **It keeps every result explainable and repeatable.**
4. **The honest cost:** each model, XGBoost especially, could probably score somewhat higher with careful tuning. These results show good standard-setting performance, not each model's ceiling.

In [10]:
start = time.time()
xgb_rounds, xgb_answers, xgb_scores, xgb_models = run_walk_forward(development, "XGBoost", recall_limit, feature_columns)
xgb_minutes = (time.time() - start) / 60
print(f"XGBoost, all 4 rounds: {xgb_minutes:.1f} minutes")
print("same pooled answers as Logistic Regression:", np.array_equal(xgb_answers, pooled_answers))
print()

# The class weight XGBoost used in each round (genuine / fraud in the learning blocks).
for round_number in range(1, 5):
    print(f"round {round_number}: scale_pos_weight {xgb_models[round_number - 1].get_params()['scale_pos_weight']:.1f}")

XGBoost, all 4 rounds: 0.0 minutes
same pooled answers as Logistic Regression: True

round 1: scale_pos_weight 318.7
round 2: scale_pos_weight 429.3
round 3: scale_pos_weight 526.9
round 4: scale_pos_weight 522.3


### All three models, side by side

Round by round on both metrics, then pooled with each model's own 95% interval. All three are bootstrapped together (`bootstrap_metrics`, 1,000 resamples, seed 42), so each resample scores them on exactly the same rows; those same resamples give the paired differences in the next step. Both tables are saved to `outputs/tables`. No winner is called here.

In [11]:
all_rounds = pd.concat([lr_rounds, rf_rounds, xgb_rounds])
all_rounds.to_csv("../outputs/tables/models_by_round.csv", index=False)

print("Low-recall AUCPR by round:")
print(all_rounds.pivot(index="model", columns="round", values="low_recall_aucpr").round(4))
print()
print("Average precision by round:")
print(all_rounds.pivot(index="model", columns="round", values="average_precision").round(4))
print()

model_scores = {"Logistic Regression": lr_scores, "Random Forest": rf_scores, "XGBoost": xgb_scores}
model_minutes = {"Logistic Regression": lr_minutes, "Random Forest": rf_minutes, "XGBoost": xgb_minutes}
model_bootstrap = bootstrap_metrics(pooled_answers, model_scores, recall_limit, 1000, 42)

pooled_rows = []
for model_name in model_scores:
    scores = model_scores[model_name]
    low_recall_list = model_bootstrap[model_name]["low_recall"]
    ap_list = model_bootstrap[model_name]["average_precision"]
    pooled_rows.append({
        "model": model_name,
        "low_recall_aucpr": aucpr_low_recall(pooled_answers, scores, recall_limit),
        "low_recall_ci_lower": np.percentile(low_recall_list, 2.5),
        "low_recall_ci_upper": np.percentile(low_recall_list, 97.5),
        "average_precision": average_precision_score(pooled_answers, scores),
        "ap_ci_lower": np.percentile(ap_list, 2.5),
        "ap_ci_upper": np.percentile(ap_list, 97.5),
        "distinct_scores": len(np.unique(scores)),
        "minutes_for_4_rounds": model_minutes[model_name],
    })

pooled_table = pd.DataFrame(pooled_rows)
pooled_table.to_csv("../outputs/tables/models_pooled.csv", index=False)
print("Pooled across all four check blocks (250 fraud):")
print(pooled_table.round(4).T)

Low-recall AUCPR by round:
round                     1       2       3       4
model                                              
Logistic Regression  0.8292  0.8854  1.0000  1.0000
Random Forest        1.0000  0.8918  0.9604  0.9917
XGBoost              1.0000  0.9192  1.0000  1.0000

Average precision by round:
round                     1       2       3       4
model                                              
Logistic Regression  0.5896  0.5823  0.8413  0.7821
Random Forest        0.8057  0.6724  0.7878  0.7482
XGBoost              0.7881  0.6745  0.7851  0.7811



Pooled across all four check blocks (250 fraud):
                                        0              1        2
model                 Logistic Regression  Random Forest  XGBoost
low_recall_aucpr                   0.9966         0.9595   0.9688
low_recall_ci_lower                0.9831         0.8958   0.9116
low_recall_ci_upper                   1.0         0.9976      1.0
average_precision                  0.6923         0.7581   0.7527
ap_ci_lower                         0.631         0.6977    0.694
ap_ci_upper                        0.7492         0.8101   0.8043
distinct_scores                    173055             93   155124
minutes_for_4_rounds                0.035          0.295   0.0489


**What this shows** (no winner called yet; paired differences come next):
- **Round by round, XGBoost is the most consistent:** on the headline metric it is best or tied for best in every round (1.0000, 0.9192, 1.0000, 1.0000), and on average precision it is never far from the best (near Random Forest early, near Logistic Regression late). The other two each have a weak half: Logistic Regression early, Random Forest late.
- **Pooled, the headline metric ranks differently:** Logistic Regression 0.9966 (0.9831 to 1.0000), XGBoost 0.9688 (0.9116 to 1.0000), Random Forest 0.9595 (0.8958 to 0.9976), even though XGBoost matched or beat Logistic Regression in every single round.
- **Why that can happen:** ties are not the cause for XGBoost (155,124 distinct scores). The likely reason is that each round's model pitches its scores differently (XGBoost's class weight alone moves from 318.7 in round 1 to 526.9 in round 3), so pooling four rounds into one ranking can place a moderately confident flag from one round above a very confident one from another. Why Logistic Regression's scores pool more cleanly was not tested.
- **Pooled average precision:** Random Forest 0.7581 (0.6977 to 0.8101) and XGBoost 0.7527 (0.6940 to 0.8043) ahead of Logistic Regression 0.6923 (0.6310 to 0.7492), consistent with trees learning combinations a straight line can't.
- **All three models' own intervals overlap on both metrics**, so nothing is settled yet.
- **Speed:** Logistic Regression 0.03 minutes, XGBoost 0.04, Random Forest 0.29 for all four rounds.

## Part 5: Is any model genuinely better? Paired differences, two ways

**1. Pooled (the rule-based decision):** for each pair of models, the difference on each resample of the pooled check rows (the same 1,000 resamples as above, so the pairing holds). A model is genuinely better only if the difference's 95% interval sits entirely above or below zero.

**2. Within each round (context):** the same test inside each round separately. Within a round all three models learned from the same blocks and were scored on the same check block, so the cross-round "score pitch" problem can't arise. Each round has fewer fraud cases (46 to 88), so these intervals are wider. Each round is bootstrapped separately (1,000 resamples, seed 42).

The pooled scores are the four rounds' scores joined in order, so a round's share is found by position: the cell counts each round's scored check rows and slices the pooled lists accordingly (`scores[start:end]` is positions start to end − 1). If two models both score 1.0 on every resample, their difference is exactly 0 throughout and the interval is just 0 to 0: no difference can be shown.

`pairs` lists the three model pairs as small lists of two names, unpacked in the loop (notebook 05, Part 1).

In [12]:
pairs = [["XGBoost", "Logistic Regression"], ["XGBoost", "Random Forest"], ["Random Forest", "Logistic Regression"]]
metric_names = ["low_recall", "average_precision"]

difference_rows = []

# 1. Pooled: reuse the three-model bootstrap from above (same resamples for every model).
for name_a, name_b in pairs:
    for metric_name in metric_names:
        differences = np.array(model_bootstrap[name_a][metric_name]) - np.array(model_bootstrap[name_b][metric_name])
        difference_rows.append({"where": "pooled", "pair (A minus B)": name_a + " minus " + name_b,
                                "metric": metric_name,
                                "ci_lower": np.percentile(differences, 2.5),
                                "ci_upper": np.percentile(differences, 97.5)})

# 2. Within each round: slice each round's rows out of the pooled lists, then bootstrap that round alone.
check_sizes = []
for round_number in range(1, 5):
    check_rows = development[(development["block"] == round_number + 1) & (~development["in_gap"])]
    check_sizes.append(len(check_rows))

start = 0
for round_number in range(1, 5):
    end = start + check_sizes[round_number - 1]
    round_answers = pooled_answers[start:end]
    round_scores = {}
    for model_name in model_scores:
        round_scores[model_name] = model_scores[model_name][start:end]
    round_bootstrap = bootstrap_metrics(round_answers, round_scores, recall_limit, 1000, 42)

    for name_a, name_b in pairs:
        for metric_name in metric_names:
            differences = np.array(round_bootstrap[name_a][metric_name]) - np.array(round_bootstrap[name_b][metric_name])
            difference_rows.append({"where": "round " + str(round_number), "pair (A minus B)": name_a + " minus " + name_b,
                                    "metric": metric_name,
                                    "ci_lower": np.percentile(differences, 2.5),
                                    "ci_upper": np.percentile(differences, 97.5)})
    start = end

print("check rows per round:", check_sizes, "| total:", sum(check_sizes), "| pooled length:", len(pooled_answers))
print()

difference_table = pd.DataFrame(difference_rows)

# Plain-words verdict for each interval.
verdicts = []
for row_number in range(len(difference_table)):
    lower = difference_table.loc[row_number, "ci_lower"]
    upper = difference_table.loc[row_number, "ci_upper"]
    if lower > 0:
        verdicts.append("A genuinely better")
    elif upper < 0:
        verdicts.append("B genuinely better")
    else:
        verdicts.append("not proven")
difference_table["verdict"] = verdicts

difference_table.to_csv("../outputs/tables/models_paired_differences.csv", index=False)
for metric_name in metric_names:
    print(f"=== {metric_name} ===")
    print(difference_table[difference_table["metric"] == metric_name].drop(columns="metric").round(4).to_string(index=False))
    print()

check rows per round: [44015, 44111, 44114, 43935] | total: 176175 | pooled length: 176175

=== low_recall ===
  where                        pair (A minus B)  ci_lower  ci_upper    verdict
 pooled       XGBoost minus Logistic Regression   -0.0842    0.0064 not proven
 pooled             XGBoost minus Random Forest   -0.0404    0.0692 not proven
 pooled Random Forest minus Logistic Regression   -0.0997    0.0019 not proven
round 1       XGBoost minus Logistic Regression    0.0000    0.3731 not proven
round 1             XGBoost minus Random Forest   -0.0000    0.0000 not proven
round 1 Random Forest minus Logistic Regression    0.0000    0.3731 not proven
round 2       XGBoost minus Logistic Regression   -0.1283    0.2060 not proven
round 2             XGBoost minus Random Forest   -0.0965    0.1679 not proven
round 2 Random Forest minus Logistic Regression   -0.1626    0.1920 not proven
round 3       XGBoost minus Logistic Regression   -0.0041    0.0000 not proven
round 3             

**What this shows** (results only; the decision follows):
- **Headline metric (low-recall AUCPR): no pair is proven different, pooled or in any single round.** All three models are tied on the main metric. Several round intervals are exactly 0 to 0 or touch 0, because two models both reach the 1.0 ceiling.
- **Average precision, pooled (the pre-declared tie-breaker):** XGBoost is genuinely better than Logistic Regression (+0.0254 to +0.0939), and so is Random Forest (+0.0293 to +0.1006). **XGBoost against Random Forest is not proven** (−0.0272 to +0.0189).
- **Average precision, within each round:** the trees are genuinely better than Logistic Regression in rounds 1 and 2; **in round 3 the order reverses**, with Logistic Regression genuinely better than both trees (XGBoost −0.1172 to −0.0091; Random Forest −0.1208 to −0.0024); in round 4 nothing is proven. XGBoost against Random Forest is never proven in any round.
- **Do the two ways agree?** Yes on the main points: all tied on the headline metric; XGBoost and Random Forest tied with each other throughout. Pooled, the trees beat Logistic Regression; round by round that advantage is real early, reverses once, and disappears at the end, so it is not stable over time.

## Verdict: model choice

**By the pre-declared rule** (headline metric first; average precision as tie-breaker when the top candidates are tied on it): all three are tied on the headline metric, so pooled average precision decides: **XGBoost = Random Forest, both genuinely ahead of Logistic Regression.** Round by round, the trees' advantage is real in rounds 1 and 2, reverses in round 3 and disappears in round 4, so it holds overall but not at every point in time.

**Carried forward: XGBoost.** XGBoost and Random Forest could not be separated by the data, so this is a **practical tie-break, not a performance win**: XGBoost's scores are smooth (155,124 distinct values against the forest's 93, which matters for choosing a decision threshold in Phase 7), it was best or tied on the headline metric in every round, it is about 6 times faster, and it is the model the project's plan names.

## Part 6: Does an hour-of-day feature help XGBoost?

The idea noted in notebook 02: fraud's share rises in the quiet night-time hours, but `Time` counts up like a stopwatch, so the same point in the daily cycle looks different on day one and day two.

`hour_of_day = (Time // 3600) % 24`: `// 3600` gives whole hours since the first transaction (0 to 47, notebook 02); **`% 24`** keeps the remainder after dividing by 24, so the count wraps around like a clock (hour 25 becomes 1, hour 28 becomes 4) and both night periods share the same values.

- **No leakage:** it is calculated from each transaction's own Time, with nothing learned from other rows, so it needs no fitting.
- **Honest limit:** hour 0 is "when the data starts", not midnight, so this is a relative clock that lines the two days up, not the real time of day.
- **One number, 0 to 23:** fine for XGBoost, which can pick out a range of hours with two questions. (The alternative, a circular encoding with two columns, sine and cosine, tells a model that hour 23 and hour 0 are neighbours; that matters for straight-line models, and the quiet period here, roughly hours 1 to 7, doesn't cross the 23 to 0 wrap.)
- It is added to the development table in this notebook only; no data file changes. `prepare_round` and `run_walk_forward` now take the list of feature columns to use, so the same code runs with 29 or 30 features.

**Where it can matter:** only round 3's check block (B4, hours 22.7 to 34.8) contains a night period (night 2), with round 3 having learned from night 1 in B1. The other check blocks are daytime or evening, where the feature can barely help. The cell below counts each check block's rows in hours 1 to 7 to confirm this.

**Decision rule, fixed before any result exists:** the burden of proof is on the new feature. It is **kept** only if the pooled paired difference (with minus without) on the headline metric sits entirely above zero, or, if that is not proven, the pooled difference on average precision sits entirely above zero. **Otherwise it is left out**, including when nothing is proven either way. Round-by-round results (especially round 3) are reported as context.

In [13]:
development["hour_of_day"] = (development["Time"] // 3600) % 24
columns_with_hour = feature_columns + ["hour_of_day"]

# How many of each check block's scored rows fall in the quiet hours 1 to 7?
for round_number in range(1, 5):
    check_rows = development[(development["block"] == round_number + 1) & (~development["in_gap"])]
    in_quiet_hours = (check_rows["hour_of_day"] >= 1) & (check_rows["hour_of_day"] <= 7)
    print(f"round {round_number} check block: {in_quiet_hours.sum():,} rows in hours 1-7"
          f" ({check_rows[in_quiet_hours]['Class'].sum()} fraud) out of {len(check_rows):,}")
print()

start = time.time()
hour_rounds, hour_answers, hour_scores, hour_models = run_walk_forward(development, "XGBoost", recall_limit, columns_with_hour)
print(f"XGBoost with hour_of_day, all 4 rounds: {(time.time() - start) / 60:.1f} minutes")
print("same pooled answers as before:", np.array_equal(hour_answers, pooled_answers))

round 1 check block: 0 rows in hours 1-7 (0 fraud) out of 44,015
round 2 check block: 0 rows in hours 1-7 (0 fraud) out of 44,111
round 3 check block: 13,957 rows in hours 1-7 (62 fraud) out of 44,114
round 4 check block: 0 rows in hours 1-7 (0 fraud) out of 43,935



XGBoost with hour_of_day, all 4 rounds: 0.1 minutes
same pooled answers as before: True


### With against without: round by round, pooled, and paired differences

Same method as Part 5: both versions bootstrapped on the same resampled rows (1,000 resamples, seed 42), pooled and within each round (slicing by `check_sizes` from Part 5). Difference = with hour_of_day minus without.

In [14]:
hour_rounds["model"] = "XGBoost with hour_of_day"
hour_comparison = pd.concat([xgb_rounds, hour_rounds])
print("Low-recall AUCPR by round:")
print(hour_comparison.pivot(index="model", columns="round", values="low_recall_aucpr").round(4))
print()
print("Average precision by round:")
print(hour_comparison.pivot(index="model", columns="round", values="average_precision").round(4))
print()

version_scores = {"without": xgb_scores, "with": hour_scores}
hour_rows = []

# Pooled.
pooled_bootstrap = bootstrap_metrics(pooled_answers, version_scores, recall_limit, 1000, 42)
for version_name in version_scores:
    print(f"pooled, {version_name:7s}: low-recall AUCPR {aucpr_low_recall(pooled_answers, version_scores[version_name], recall_limit):.4f}"
          f"  average precision {average_precision_score(pooled_answers, version_scores[version_name]):.4f}")
for metric_name in metric_names:
    differences = np.array(pooled_bootstrap["with"][metric_name]) - np.array(pooled_bootstrap["without"][metric_name])
    hour_rows.append({"where": "pooled", "metric": metric_name,
                      "ci_lower": np.percentile(differences, 2.5), "ci_upper": np.percentile(differences, 97.5)})

# Within each round (same slicing as Part 5).
start = 0
for round_number in range(1, 5):
    end = start + check_sizes[round_number - 1]
    round_scores = {"without": xgb_scores[start:end], "with": hour_scores[start:end]}
    round_bootstrap = bootstrap_metrics(pooled_answers[start:end], round_scores, recall_limit, 1000, 42)
    for metric_name in metric_names:
        differences = np.array(round_bootstrap["with"][metric_name]) - np.array(round_bootstrap["without"][metric_name])
        hour_rows.append({"where": "round " + str(round_number), "metric": metric_name,
                          "ci_lower": np.percentile(differences, 2.5), "ci_upper": np.percentile(differences, 97.5)})
    start = end

hour_table = pd.DataFrame(hour_rows)
hour_verdicts = []
for row_number in range(len(hour_table)):
    if hour_table.loc[row_number, "ci_lower"] > 0:
        hour_verdicts.append("with genuinely better")
    elif hour_table.loc[row_number, "ci_upper"] < 0:
        hour_verdicts.append("without genuinely better")
    else:
        hour_verdicts.append("not proven")
hour_table["verdict"] = hour_verdicts

hour_table.to_csv("../outputs/tables/hour_of_day_paired_differences.csv", index=False)
print()
print("With hour_of_day minus without:")
print(hour_table.round(4).to_string(index=False))

Low-recall AUCPR by round:
round                       1       2    3    4
model                                          
XGBoost                   1.0  0.9192  1.0  1.0
XGBoost with hour_of_day  1.0  0.9460  1.0  1.0

Average precision by round:
round                          1       2       3       4
model                                                   
XGBoost                   0.7881  0.6745  0.7851  0.7811
XGBoost with hour_of_day  0.7964  0.6863  0.7902  0.7841



pooled, without: low-recall AUCPR 0.9688  average precision 0.7527
pooled, with   : low-recall AUCPR 0.9736  average precision 0.7602



With hour_of_day minus without:
  where            metric  ci_lower  ci_upper               verdict
 pooled        low_recall   -0.0156    0.0281            not proven
 pooled average_precision   -0.0023    0.0175            not proven
round 1        low_recall   -0.0000    0.0000            not proven
round 1 average_precision    0.0001    0.0240 with genuinely better
round 2        low_recall   -0.0070    0.0997            not proven
round 2 average_precision   -0.0041    0.0329            not proven
round 3        low_recall   -0.0285    0.0000            not proven
round 3 average_precision   -0.0136    0.0257            not proven
round 4        low_recall   -0.0000    0.0000            not proven
round 4 average_precision   -0.0109    0.0208            not proven


**What this shows, and the decision:**
- **Only round 3's check block contains night-time hours:** 13,957 rows in hours 1 to 7, holding 62 of its 88 fraud cases; the other three check blocks have none.
- **With hour_of_day, XGBoost scores slightly higher almost everywhere** (pooled low-recall AUCPR 0.9736 against 0.9688; average precision 0.7602 against 0.7527), but **neither pooled paired difference is proven** (low-recall −0.0156 to +0.0281; average precision −0.0023 to +0.0175).
- **Round 3, the real test, shows nothing:** both versions hit the 1.0 ceiling on the headline metric, and the average-precision difference could go either way (−0.0136 to +0.0257). One untested reading: the V columns already carry the night-time signal.
- **Round 1's single "genuinely better" result is not trusted:** its check block has no night hours, so it can't be the effect being tested, and among 10 intervals one landing just clear of zero (lower end +0.0001) is what luck alone could produce.
- **Decision, by the rule fixed before running: hour_of_day is left out.** The night-time rise in fraud's share is real (notebook 02), but as a feature it did not measurably improve XGBoost here.